# PBI-Scope Database Exploration

This notebook demonstrates how to explore the PBI-Scope database using both **Python** and **R**.

**Prerequisites:**
- The custom R+Python container must be running
- The `pbi-data` Docker volume must be mounted at `/data`

**Setup:**
```bash
cd mount_scripts/
docker compose -f docker-compose.custom.yml build
docker compose -f docker-compose.custom.yml up custom-jupyter
```

Then open `http://localhost:8888` in your browser.

---
## Python Exploration

The `pbi` package provides a convenient Python interface for accessing the PBI-Scope database.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# The pbi package provides quick_connect() for easy database access
from pbi import quick_connect

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Connect to database
retriever = quick_connect()
print("Connected to database!")
print(f"Database path: {retriever.db_path}")

### Python: Source Database Distribution

Which public databases contribute the most phages?

In [ ]:
# Query source distribution
source_dist = retriever.conn.execute("""
    SELECT Source_DB, COUNT(*) AS phage_count
    FROM fact_phages
    GROUP BY Source_DB
    ORDER BY phage_count DESC
""").fetchdf()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
source_dist.plot(kind="barh", x="Source_DB", y="phage_count", ax=ax, legend=False)
ax.set_title("Phages by Source Database")
ax.set_xlabel("Number of Phages")
ax.set_ylabel(None)
plt.tight_layout()
fig.savefig(output_dir / "python_source_distribution.png", dpi=150)
plt.show()

### Python: Genome Length Distribution

Phage genome lengths vary widely (from ~2 kb to ~500 kb).

In [ ]:
# Query genome lengths
lengths = retriever.conn.execute("""
    SELECT Length
    FROM fact_phages
    WHERE Length > 0 AND Length < 200000
""").fetchdf()

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
lengths["Length"].div(1000).hist(bins=50, ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Phage Genome Length Distribution")
ax.set_xlabel("Genome Length (kb)")
ax.set_ylabel("Count")
plt.tight_layout()
fig.savefig(output_dir / "python_length_distribution.png", dpi=150)
plt.show()

---
## R Exploration

R connects directly to the DuckDB database using the `DBI` and `duckdb` packages.
No special `pbi` package is needed for R.

In [ ]:
# Load R libraries
library(DBI)
library(duckdb)
library(ggplot2)
library(scales)

In [ ]:
# Connect to database
db_path <- file.path(Sys.getenv("DATA_PATH", "/data/processed"),
                     "databases", "phage_database_optimized.duckdb")
con <- dbConnect(duckdb(), dbdir = db_path, read_only = TRUE)

# List tables
cat("Available tables:", paste(dbListTables(con), collapse = ", "), "\n")

### R: Lifestyle Distribution

Phages are classified as virulent (lytic) or temperate (lysogenic).

In [ ]:
# Query lifestyle distribution
lifestyle <- dbGetQuery(con, "
    SELECT Lifestyle, COUNT(*) AS count
    FROM fact_phages
    WHERE Lifestyle IN ('virulent', 'temperate')
    GROUP BY Lifestyle
")

# Plot
ggplot(lifestyle, aes(x = Lifestyle, y = count, fill = Lifestyle)) +
    geom_bar(stat = "identity") +
    scale_y_continuous(labels = comma) +
    scale_fill_manual(values = c("virulent" = "#E74C3C", "temperate" = "#3498DB")) +
    labs(
        title = "Phage Lifestyle Distribution (R)",
        x = NULL,
        y = "Count"
    ) +
    theme_minimal(base_size = 12) +
    theme(legend.position = "none")

### R: GC Content Comparison

Comparing GC content between phages and their hosts can reveal co-evolution patterns.

In [ ]:
# Query GC content
gc_comparison <- dbGetQuery(con, "
    SELECT 
        'Phage' AS type,
        GC_content AS GC
    FROM fact_phages
    WHERE GC_content > 0 AND GC_content < 100
    UNION ALL
    SELECT
        'Host' AS type,
        GC_Content AS GC
    FROM dim_hosts
    WHERE GC_Content > 0 AND GC_Content < 100
")

# Plot
ggplot(gc_comparison, aes(x = GC, fill = type)) +
    geom_density(alpha = 0.5) +
    scale_fill_manual(values = c("Phage" = "#E74C3C", "Host" = "#3498DB")) +
    labs(
        title = "GC Content Distribution: Phages vs Hosts (R)",
        x = "GC Content (%)",
        y = "Density",
        fill = "Type"
    ) +
    theme_minimal(base_size = 12)

---
## Cleanup

Always close the database connection when done.

In [ ]:
# Python cleanup
retriever.close()
print("Python connection closed.")

In [ ]:
# R cleanup
dbDisconnect(con, shutdown = TRUE)
cat("R connection closed.\n")

---
## Running as a Script

For long-running analyses, you can run the scripts directly instead of using Jupyter:

```bash
# Run the R script
docker compose -f docker-compose.custom.yml up custom-scripts

# Or run the Python script
docker compose -f docker-compose.custom.yml run custom-jupyter python explore_phages.py
```

Plots are saved to the `output/` directory on your host machine.